In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE_PATH = "/content/drive/MyDrive/your folder"
DDR_ZIP = "/content/drive/MyDrive/DDR.zip"          # change to your DDR zip path
if not os.path.exists("/content/ddr/DR_grading.csv"):
    os.system(f'unzip -q "{DDR_ZIP}" -d /content/ddr')
print(os.listdir("/content/ddr"))

Mounted at /content/drive
['DR_grading.csv', 'DR_grading']


In [ ]:
import os, hashlib, numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from sklearn.model_selection import train_test_split

DDR_DIR = "/content/ddr"
IMG_EXT = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")

# ---- 1. read the label csv and find every image ----
by_name = {}
for root, _, files in os.walk(DDR_DIR):
    for f in files:
        if f.lower().endswith(IMG_EXT): by_name.setdefault(f, []).append(os.path.join(root, f))
lab = pd.read_csv(os.path.join(DDR_DIR, "DR_grading.csv"))[["id_code", "diagnosis"]].copy()
lab["csv_row"] = np.arange(len(lab))        # position in the CSV = position in the saved ext_ddr_all_*.npz arrays
lab["path"] = lab.id_code.map(lambda n: by_name[n][0] if n in by_name else None)
print("labels:", len(lab), "| without image:", int(lab.path.isna().sum()))
lab = lab[lab.path.notna()].copy()

# ---- 2. fingerprint every file (exact copies only) ----
def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()
with ThreadPoolExecutor(4) as ex:
    lab["md5"] = list(tqdm(ex.map(md5, lab.path), total=len(lab)))

# ---- 3. duplicates inside DDR ----
grp = lab.groupby("md5").diagnosis.agg(["size", "nunique"])
dup_groups = grp[grp["size"] > 1]
conf_groups = grp[grp["nunique"] > 1]
print("exact-duplicate groups inside DDR:", len(dup_groups), "| images involved:", int(dup_groups["size"].sum()),
      "| groups with conflicting grades:", len(conf_groups))

# ---- 4. exact copies of APTOS images (hashes saved earlier in aptos_clean_split.csv) ----
ap = pd.read_csv(BASE_PATH + "/aptos_clean_split.csv")
aptos_md5 = set(ap["md5"]) if "md5" in ap.columns else set()
print("APTOS hashes available:", len(aptos_md5))
lab["in_aptos"] = lab.md5.isin(aptos_md5)
print("DDR images that are byte-identical to an APTOS image:", int(lab.in_aptos.sum()))

# ---- 5. clean: one copy per group, drop conflicting groups and APTOS copies ----
keep, removed_dup, removed_conf = [], 0, 0
for h, g in lab.groupby("md5"):
    if len(g) == 1: keep.append(g.index[0]); continue
    if g.diagnosis.nunique() > 1: removed_conf += len(g); continue
    keep.append(g.index[0]); removed_dup += len(g) - 1
clean = lab.loc[sorted(keep)]
n_before = len(clean); clean = clean[~clean.in_aptos]
print(f"\nDDR images: {len(lab)} | duplicate copies removed: {removed_dup} | removed (conflicting grades): {removed_conf} | "
      f"removed (copy of an APTOS image): {n_before - len(clean)} | kept: {len(clean)}")

# ---- 6. our own split (70/15/15, stratified by grade, fixed seed) ----
SEED = 42
train_df, tmp = train_test_split(clean, test_size=0.30, stratify=clean.diagnosis, random_state=SEED)
val_df, test_df = train_test_split(tmp, test_size=0.50, stratify=tmp.diagnosis, random_state=SEED)
clean = pd.concat([train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")])
for nm, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{nm:5s} {len(d):6d}", np.bincount(d.diagnosis, minlength=5))

# ---- 7. save (file names and labels only, no runtime paths) ----
lab.drop(columns="path").to_csv(BASE_PATH + "/ddr_audit.csv", index=False)
clean[["id_code", "diagnosis", "split", "md5", "csv_row"]].to_csv(BASE_PATH + "/ddr_clean_split.csv", index=False)
print("\nsaved ddr_audit.csv and ddr_clean_split.csv to", BASE_PATH)

labels: 12522 | without image: 0


100%|██████████| 12522/12522 [00:10<00:00, 1251.15it/s]


exact-duplicate groups inside DDR: 95 | images involved: 193 | groups with conflicting grades: 0
APTOS hashes available: 3504
DDR images that are byte-identical to an APTOS image: 0

DDR images: 12522 | duplicate copies removed: 98 | removed (conflicting grades): 0 | removed (copy of an APTOS image): 0 | kept: 12424
train   8696 [4317  441 3134  165  639]
val     1864 [925  95 671  36 137]
test    1864 [926  94 672  35 137]

saved ddr_audit.csv and ddr_clean_split.csv to /content/drive/MyDrive/APTOS-Finalkefinal(modified-NEW1)


In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, roc_auc_score
BASE_PATH = "/content/drive/MyDrive/your folder"
cs = pd.read_csv(BASE_PATH + "/ddr_clean_split.csv")
sets = {"DDR all (deduplicated)": cs.csv_row.values, "DDR held-out test split": cs[cs.split == "test"].csv_row.values}
runs = [("circuit", f"v3_circuit_aNone_pe0_L4_s{s}") for s in (42, 43, 44)] + \
       [("skip", f"skip50_skip_aNone_pe0_L4_s{s}") for s in (42, 43, 44)]
rows = []
for name, idx in sets.items():
    for fam, r in runs:
        d = np.load(f"{BASE_PATH}/ext_ddr_all_{r}.npz"); P, Y = d["P"][idx], d["Y"][idx]; pr = P.argmax(1)
        rows.append(dict(set=name, family=fam, n=len(Y), acc=accuracy_score(Y, pr), f1=f1_score(Y, pr, average="macro"),
                         qwk=cohen_kappa_score(Y, pr, weights="quadratic"),
                         auc=roc_auc_score(Y, P, multi_class="ovr"), ref_auc=roc_auc_score((Y >= 2).astype(int), P[:, 2:].sum(1))))
res = pd.DataFrame(rows)
print(res.groupby(["set", "family"])[["n", "acc", "f1", "qwk", "auc", "ref_auc"]].agg(["mean", "std"]).round(3).to_string())

                                       n         acc            f1           qwk           auc        ref_auc       
                                    mean  std   mean    std   mean    std   mean    std   mean    std    mean    std
set                     family                                                                                      
DDR all (deduplicated)  circuit  12424.0  0.0  0.618  0.025  0.371  0.013  0.566  0.031  0.736  0.023   0.786  0.026
                        skip     12424.0  0.0  0.627  0.022  0.387  0.015  0.572  0.040  0.739  0.014   0.807  0.030
DDR held-out test split circuit   1864.0  0.0  0.620  0.029  0.374  0.027  0.563  0.034  0.719  0.025   0.783  0.033
                        skip      1864.0  0.0  0.635  0.021  0.394  0.009  0.576  0.035  0.730  0.017   0.805  0.036


In [ ]:
IDRID_ZIP = "/content/drive/MyDrive/your folder.zip"          # change to your IDRiD zip path
if not os.path.exists("/content/idrid/IDRID"):
    os.system(f'unzip -q "{IDRID_ZIP}" -d /content/idrid')
idr = [os.path.join(r, f) for r, _, fs in os.walk("/content/idrid/IDRID/1. Original Images") for f in fs
       if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff"))]
idr_md5 = {md5(p) for p in idr}
print(len(idr), "IDRiD images | identical to a DDR image:", len(idr_md5 & set(lab.md5)), "| identical to an APTOS image:", len(idr_md5 & aptos_md5))

516 IDRiD images | identical to a DDR image: 0 | identical to an APTOS image: 0
